
# Exploratory Data Analysis of House Prices

## Project Overview

The objective of this project is to perform Exploratory Data Analysis (EDA) on the Ames Housing Dataset.

The project focuses on understanding different factors that affect house sale prices. We will analyse numerical and categorical data, calculate statistical measures, identify missing values and outliers, and create visualizations.

## Dataset Description

The dataset contains information about houses, including their area, quality, neighborhood, year of construction, and sale price.

## Objectives

1. Understand the distribution of house prices.
2. Identify features correlated with SalePrice.
3. Analyse the effect of overall quality on house prices.
4. Compare house prices across neighborhoods.
5. Detect outliers and missing values.
6. Create visualizations and a final dashboard.


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("All libraries imported successfully!")


All libraries imported successfully!



## 1. Data Loading

In this section, we upload the Ames Housing Dataset and load it using Pandas. We will use the dataset to perform exploratory data analysis.


In [ ]:

from google.colab import files

uploaded = files.upload()


In [ ]:

df = pd.read_csv("/content/train.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

df.head()



## 2. Data Understanding

In this section, we will examine the dataset, understand its structure, identify data types and check for missing values.


In [ ]:

print("Dataset Shape:")
print(df.shape)

print("\nFirst 5 Rows:")
display(df.head())

print("\nDataset Information:")
df.info()


In [ ]:

missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

missing_percentage = (missing / len(df)) * 100

missing_table = pd.DataFrame({
    "Missing Values": missing,
    "Percentage": missing_percentage
})

display(missing_table)
print("Total columns with missing values:", len(missing))



### Observation

The dataset contains approximately 1460 rows and 81 columns. It includes numerical and categorical information about houses. Some columns contain missing values, which need to be examined before further analysis.



## 3. Variable Classification

In this section, we classify the columns into numerical, categorical and ordinal variables based on their data types and meaning.


In [ ]:

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

categorical_cols = df.select_dtypes(include="object").columns.tolist()

print("Numerical Columns:")
print(numeric_cols)

print("\nCategorical Columns:")
print(categorical_cols)

print("\nNumber of Numerical Columns:", len(numeric_cols))
print("Number of Categorical Columns:", len(categorical_cols))


In [ ]:

ordinal_map = {
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"]
}

for column, order in ordinal_map.items():
    df[column] = pd.Categorical(
        df[column],
        categories=order,
        ordered=True
    )

print("Ordinal variables identified successfully!")
print("OverallQual is also an ordinal variable (1 to 10).")


In [ ]:

variable_summary = []

for col in df.columns:
    if col in ["OverallQual", "ExterQual", "KitchenQual"]:
        variable_type = "Ordinal"
    elif col in numeric_cols:
        variable_type = "Numerical"
    else:
        variable_type = "Categorical"

    variable_summary.append({
        "Column": col,
        "Type": variable_type,
        "Unique Values": df[col].nunique(),
        "Example Values": str(df[col].dropna().unique()[:3])
    })

variable_table = pd.DataFrame(variable_summary)

display(variable_table.head(20))



### Observation

The dataset contains numerical, categorical and ordinal variables. Numerical variables include SalePrice and GrLivArea. Categorical variables include Neighborhood and HouseStyle. OverallQual is an ordinal variable because it represents house quality on a scale from 1 to 10.



## 4. Descriptive Statistics

Descriptive statistics help us understand the central values and spread of the dataset. We will calculate different statistical measures for SalePrice, GrLivArea and LotArea.


In [ ]:

display(df.describe())


In [ ]:

columns = ["SalePrice", "GrLivArea", "LotArea"]

for col in columns:
    print("\nStatistics for:", col)

    print("Mean:", df[col].mean())
    print("Median:", df[col].median())
    print("Mode:", df[col].mode()[0])
    print("Standard Deviation:", df[col].std())
    print("Variance:", df[col].var())
    print("Range:", df[col].max() - df[col].min())
    print("IQR:", df[col].quantile(0.75) - df[col].quantile(0.25))
    print("Skewness:", df[col].skew())
    print("Kurtosis:", df[col].kurt())


In [ ]:

print("Neighborhood Value Counts:")
display(df["Neighborhood"].value_counts())

print("\nHouseStyle Value Counts:")
display(df["HouseStyle"].value_counts())

print("\nNeighborhood Mode:", df["Neighborhood"].mode()[0])
print("HouseStyle Mode:", df["HouseStyle"].mode()[0])



### Observation

Neighborhood and HouseStyle contain different categories of houses. The value counts show how frequently each category appears in the dataset. The mode represents the most frequently occurring category.




## 5. Data Cleaning

In this section, we will check duplicate records, handle missing values, standardize categorical values and identify impossible values.


In [ ]:

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

df = df.drop_duplicates()

print("Dataset Shape After Removing Duplicates:", df.shape)


In [ ]:

# Columns where missing values mean the feature is absent

absent_columns = [
    "PoolQC", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual",
    "GarageCond", "BsmtQual", "BsmtCond",
    "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType", "MiscFeature"
]

# Replace missing values with None
for col in absent_columns:
    if col in df.columns:
        df[col] = df[col].fillna("None")

# Fill remaining categorical missing values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("Unknown")

# Handle missing values in ordinal columns
for col in df.select_dtypes(include="category").columns:
    if df[col].isnull().any():
        df[col] = df[col].cat.add_categories(
            ["Unknown"]
        ).fillna("Unknown")

# Fill numerical missing values using median
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

print("Missing values handled successfully!")


In [ ]:

print("Total Missing Values:", df.isnull().sum().sum())

display(df.isnull().sum().sort_values(ascending=False).head(10))


In [ ]:

print("Houses built after the year of sale:")
display(df[df["YearBuilt"] > df["YrSold"]])

print("Negative LotArea values:", (df["LotArea"] < 0).sum())
print("Negative GrLivArea values:", (df["GrLivArea"] < 0).sum())



### Observation

Duplicate records were checked and removed if present. Missing values were handled using suitable methods. Some missing categorical values represent the absence of a feature rather than unknown information. Numerical missing values were filled using the median.



## 6. Univariate Analysis

Univariate analysis studies one variable at a time. We will examine the distribution of SalePrice, house quality and house styles using different graphs.


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram with KDE
sns.histplot(df["SalePrice"], kde=True, ax=axes[0])
axes[0].set_title("SalePrice Distribution")
axes[0].set_xlabel("Sale Price")

# Boxplot
sns.boxplot(x=df["SalePrice"], ax=axes[1])
axes[1].set_title("SalePrice Boxplot")

# Log transformed histogram
sns.histplot(np.log1p(df["SalePrice"]), kde=True, ax=axes[2])
axes[2].set_title("Log Transformed SalePrice")
axes[2].set_xlabel("Log SalePrice")

plt.tight_layout()
plt.show()

print("Original Skewness:", df["SalePrice"].skew())
print("Log Skewness:", np.log1p(df["SalePrice"]).skew())


In [ ]:

plt.figure(figsize=(10, 6))

sns.boxplot(data=df, x="OverallQual", y="SalePrice")

plt.title("House Prices by Overall Quality")
plt.xlabel("Overall Quality")
plt.ylabel("Sale Price")

plt.show()


In [ ]:

neighborhood_order = (
    df.groupby("Neighborhood", observed=True)["SalePrice"]
    .median()
    .sort_values()
    .index
)

plt.figure(figsize=(14, 6))

sns.boxplot(
    data=df,
    x="Neighborhood",
    y="SalePrice",
    order=neighborhood_order
)

plt.title("House Prices by Neighborhood")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x="OverallQual", ax=axes[0])
axes[0].set_title("Count of Overall Quality")
axes[0].set_xlabel("Overall Quality")

sns.countplot(data=df, x="HouseStyle", ax=axes[1])
axes[1].set_title("Count of House Styles")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()



### Observation

The SalePrice distribution shows how house prices are spread across the dataset. The histogram and skewness values help us identify whether the distribution is symmetrical or skewed. The boxplot helps identify unusual price values. We can also compare prices across different quality levels and neighborhoods.



## 7. Correlation and Covariance

Correlation measures the strength and direction of the relationship between two numerical variables. Covariance shows how two variables change together.


In [ ]:

numeric_data = df.select_dtypes(include=np.number).drop(
    columns=["Id"], errors="ignore"
)

corr_matrix = numeric_data.corr()
cov_matrix = numeric_data.cov()

print("Correlation Matrix:")
display(corr_matrix)

print("Covariance Matrix:")
display(cov_matrix)


In [ ]:

top_corr = (
    corr_matrix["SalePrice"]
    .drop("SalePrice")
    .abs()
    .sort_values(ascending=False)
    .head(10)
)

print("Top 10 Features Correlated with SalePrice:")
display(top_corr)

print("\nActual Correlation Values:")
display(corr_matrix.loc[top_corr.index, "SalePrice"])


In [ ]:

top3 = top_corr.index[:3]

for col in top3:
    print("\nFeature:", col)
    print("Correlation with SalePrice:",
          corr_matrix.loc[col, "SalePrice"])
    print("Covariance with SalePrice:",
          cov_matrix.loc[col, "SalePrice"])



### Observation

Correlation helps identify features that have a strong relationship with SalePrice. Its values range from -1 to +1, making comparisons easier. Covariance depends on the units of the variables, so it is harder to compare directly. Correlation does not necessarily mean causation.



## 8. Scatter Plots and Pair Plots

Scatter plots help us understand relationships between two numerical variables. Pair plots allow us to examine multiple relationships together.


In [ ]:

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="GrLivArea",
    y="SalePrice"
)

plt.title("Living Area vs Sale Price")
plt.xlabel("Above Ground Living Area")
plt.ylabel("Sale Price")

plt.show()


In [ ]:

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="TotalBsmtSF",
    y="SalePrice"
)

plt.title("Basement Area vs Sale Price")
plt.xlabel("Total Basement Area")
plt.ylabel("Sale Price")

plt.show()


In [ ]:

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="YearBuilt",
    y="SalePrice"
)

plt.title("Year Built vs Sale Price")
plt.xlabel("Year Built")
plt.ylabel("Sale Price")

plt.show()


In [ ]:

selected = [
    "SalePrice",
    "GrLivArea",
    "TotalBsmtSF",
    "GarageArea",
    "YearBuilt"
]

sns.pairplot(
    df[selected].sample(
        n=min(500, len(df)),
        random_state=42
    ),
    diag_kind="kde"
)

plt.show()



### Observation

Scatter plots help us understand how living area, basement area and construction year are related to house prices. Pair plots show relationships between multiple numerical features. Generally, larger living areas tend to be associated with higher house prices.



## 9. Correlation Heatmap

A correlation heatmap provides a visual representation of relationships between numerical variables. It helps identify positively and negatively correlated features.


In [ ]:

top10_features = [
    "SalePrice"
] + top_corr.index.tolist()

corr_top = numeric_data[top10_features].corr()

mask = np.triu(
    np.ones_like(corr_top, dtype=bool),
    k=1
)

plt.figure(figsize=(12, 9))

sns.heatmap(
    corr_top,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    mask=mask
)

plt.title("Top 10 Features Correlated with SalePrice")
plt.tight_layout()
plt.show()



### Observation

The heatmap displays the correlation between the top features related to SalePrice. Values closer to +1 indicate a strong positive relationship, while values closer to -1 indicate a strong negative relationship. Values near zero indicate a weak linear relationship.



## 10. Outlier Detection

In this section, we identify outliers using the IQR method and Z-score method. We will compare the results to understand unusual house prices and living areas.


In [ ]:

def iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return series[(series < lower) | (series > upper)]

print("IQR Outliers in GrLivArea:",
      len(iqr_outliers(df["GrLivArea"])))

print("IQR Outliers in SalePrice:",
      len(iqr_outliers(df["SalePrice"])))


In [ ]:

def zscore_outliers(series, threshold=3):
    z = np.abs(stats.zscore(series))
    return series[z > threshold]

print("Z-score Outliers in GrLivArea:",
      len(zscore_outliers(df["GrLivArea"])))

print("Z-score Outliers in SalePrice:",
      len(zscore_outliers(df["SalePrice"])))


In [ ]:

for col in ["GrLivArea", "SalePrice"]:

    iqr_result = iqr_outliers(df[col])
    z_result = zscore_outliers(df[col])

    print("\nColumn:", col)
    print("IQR Outliers:", len(iqr_result))
    print("Z-score Outliers:", len(z_result))
    print("Common Outliers:", len(
        set(iqr_result.index) & set(z_result.index)
    ))


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(y=df["GrLivArea"], ax=axes[0])
axes[0].set_title("GrLivArea Outliers")

sns.boxplot(y=df["SalePrice"], ax=axes[1])
axes[1].set_title("SalePrice Outliers")

plt.tight_layout()
plt.show()


In [ ]:

outlier_index = set(iqr_outliers(df["GrLivArea"]).index)

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="GrLivArea",
    y="SalePrice",
    color="blue",
    label="Normal"
)

outlier_df = df.loc[list(outlier_index)]

plt.scatter(
    outlier_df["GrLivArea"],
    outlier_df["SalePrice"],
    color="red",
    label="IQR Outliers"
)

plt.title("Outlier Detection in House Prices")
plt.xlabel("Living Area")
plt.ylabel("Sale Price")
plt.legend()

plt.show()



### Observation

The IQR and Z-score methods identify unusual values using different approaches. IQR uses quartiles, while Z-score uses the mean and standard deviation. On skewed data, IQR may identify more outliers than Z-score. Outliers should not automatically be removed because some may represent genuine house sales.



## 11. Final Dashboard

The final dashboard combines important visualizations to summarize the analysis of house prices. It includes a line plot, bar plot, histogram and scatter plot.


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Average SalePrice by YearBuilt
year_price = df.groupby("YearBuilt")["SalePrice"].mean()

axes[0, 0].plot(
    year_price.index,
    year_price.values,
    label="Average Price"
)

axes[0, 0].set_title("Average House Price by Year Built")
axes[0, 0].set_xlabel("Year Built")
axes[0, 0].set_ylabel("Average Sale Price")
axes[0, 0].legend()

# Panel 2: Top 10 Neighborhoods
neighborhood_price = (
    df.groupby("Neighborhood")["SalePrice"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

axes[0, 1].barh(
    neighborhood_price.index[::-1],
    neighborhood_price.values[::-1]
)

axes[0, 1].set_title("Top 10 Neighborhoods by Average Price")
axes[0, 1].set_xlabel("Average Sale Price")
axes[0, 1].set_ylabel("Neighborhood")

# Panel 3: SalePrice Distribution
sns.histplot(
    df["SalePrice"],
    kde=True,
    ax=axes[1, 0]
)

axes[1, 0].axvline(
    df["SalePrice"].mean(),
    color="red",
    linestyle="--",
    label="Mean"
)

axes[1, 0].axvline(
    df["SalePrice"].median(),
    color="green",
    linestyle="--",
    label="Median"
)

axes[1, 0].set_title("SalePrice Distribution")
axes[1, 0].legend()

# Panel 4: Living Area vs SalePrice
sns.scatterplot(
    data=df,
    x="GrLivArea",
    y="SalePrice",
    hue="OverallQual",
    palette="viridis",
    ax=axes[1, 1]
)

axes[1, 1].set_title("Living Area vs SalePrice")
axes[1, 1].set_xlabel("Living Area")
axes[1, 1].set_ylabel("Sale Price")

# Final dashboard title
fig.suptitle(
    "House Price EDA Summary",
    fontsize=18
)

plt.tight_layout()

# Save dashboard
plt.savefig("eda_dashboard.png", dpi=300, bbox_inches="tight")

plt.show()

print("Dashboard created and saved successfully!")



# 12. Conclusion

In this project, we performed Exploratory Data Analysis on the Ames Housing Dataset using Python.

We studied house prices, analysed numerical and categorical variables, calculated descriptive statistics, identified missing values and detected outliers.

The analysis helped us understand the relationship between house features and sale prices.

### Key Findings

1. SalePrice distribution was analysed using histograms and boxplots.
2. OverallQual and GrLivArea were examined to understand their relationship with house prices.
3. House prices were compared across different neighborhoods.
4. Missing values were handled using suitable methods.
5. Outliers were identified using IQR and Z-score methods.
6. Different visualizations were created to understand the dataset.

### Final Result

This project helped us understand how Exploratory Data Analysis can be used to study real-world datasets, identify patterns and discover useful insights without using machine learning.
